# 03 · Macroeconomía del portafolio / mercado
La capa macro tiene dos niveles: **endógeno** (todo lo que Medallio observa entre proyectos) y **externo** (BCRP, tasa hipotecaria, TC, inflación, actividad, desempleo).

El sistema funciona aunque la tabla macro externa esté vacía; nunca rellena indicadores externos con valores inventados.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel, load_macro_inputs, build_market_panel
install_feature_mart(); panel = load_monthly_panel(); macro = load_macro_inputs(); market = build_market_panel(panel)
market.tail(18)

## Oferta agregada, demanda agregada y velocidad
En este producto, demanda agregada no significa PBI: es la demanda inmobiliaria **revelada en el portafolio** mediante separaciones netas.

In [ ]:
fig, ax = plt.subplots(figsize=(12,5))
ax.plot(market['periodo_mes'], market['stock_inicio'], label='Oferta: stock inicio')
ax.plot(market['periodo_mes'], market['demanda_neta']*5, label='Demanda neta ×5 (escala visual)')
ax.plot(market['periodo_mes'], market['saldo_final'], label='Saldo final')
ax.set_title('Portafolio · oferta vs demanda revelada'); ax.legend(); ax.grid(alpha=.2); plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12,4))
ax.plot(market['periodo_mes'], market['absorcion_neta_market'], label='Absorción neta market')
ax.axhline(market['absorcion_neta_market'].median(), linestyle='--', label='Mediana histórica')
ax.set_title('Ciclo de absorción del portafolio'); ax.legend(); ax.grid(alpha=.2); plt.show()

## Macro externo
`analytics.econ_macro_monthly` acepta tasa BCRP, tasa hipotecaria, TC PEN/USD, inflación, actividad y desempleo. Cuando haya datos, analizar rezagos: un shock de tasa puede impactar leads, separaciones y minutas con distinta demora.

In [ ]:
macro.tail(24) if not macro.empty else pd.DataFrame({'estado':['SIN INPUT MACRO EXTERNO: el sistema usa proxies endógenos hasta cargar fuentes']})

In [ ]:
if not macro.empty:
    joined = market.merge(macro, on='periodo_mes', how='inner')
    cols = ['demanda_neta','absorcion_neta_market','tasa_referencia_bcrp','tasa_hipotecaria','tc_usd_pen','inflacion_yoy','actividad_yoy']
    display(joined[cols].corr(numeric_only=True).round(3))
else:
    print('Gate: cargar macro externo antes de interpretar correlaciones con tasa/TC/inflación.')

### Lectura PMO
- oferta sube + demanda estable → riesgo de mayor tiempo de agotamiento;
- demanda cae con stock estable → revisar pricing, canal y macro;
- minutas caen pero separaciones no → problema de conversión/financiamiento, no de demanda primaria;
- correlación macro no implica causalidad: usar rezagos, controles y eventos identificables.